In [ ]:
strain_pct_cols = [
    c for c in df.columns
    if (
        "strain_plus_" in c.lower()
        and c.lower().endswith("_strain_pct")
    )
]

strain_rate_cols = [
    c for c in df.columns
    if (
        "strain_plus_" in c.lower()
        and c.lower().endswith("_strain_rate_1_s")
    )
]

print("strain pct cols:", len(strain_pct_cols))
for c in strain_pct_cols:
    print(c)

print("\nstrain rate cols:", len(strain_rate_cols))
for c in strain_rate_cols:
    print(c)

def calculate_cv_pct(data, cols):
    x = data[cols].apply(pd.to_numeric, errors="coerce")

    mean = x.mean(axis=1, skipna=True)
    sd = x.std(axis=1, skipna=True)

    cv = sd / mean.abs() * 100

    return cv.replace([np.inf, -np.inf], np.nan)

diagnostic_df = df.copy()

diagnostic_df["new_strain_plus_cv_pct"] = calculate_cv_pct(
    diagnostic_df,
    strain_pct_cols,
)

diagnostic_df["new_strain_rate_cv_pct"] = calculate_cv_pct(
    diagnostic_df,
    strain_rate_cols,
)

display(
    diagnostic_df[
        [
            "echo_lv_strain_plus_cv_pct",
            "new_strain_plus_cv_pct",
            "echo_lv_strain_rate_cv_pct",
            "new_strain_rate_cv_pct",
        ]
    ].describe().T
)

display(
    diagnostic_df[
        [
            "echo_lv_strain_plus_cv_pct",
            "new_strain_plus_cv_pct",
            "echo_lv_strain_rate_cv_pct",
            "new_strain_rate_cv_pct",
        ]
    ].head(20)
) 

strain pct cols: 18
strain_plus_basal_anteroseptal_strain_pct
strain_plus_basal_anterior_strain_pct
strain_plus_basal_anterolateral_strain_pct
strain_plus_basal_inferolateral_strain_pct
strain_plus_basal_inferior_strain_pct
strain_plus_basal_inferoseptal_strain_pct
strain_plus_mid_anteroseptal_strain_pct
strain_plus_mid_anterior_strain_pct
strain_plus_mid_anterolateral_strain_pct
strain_plus_mid_inferolateral_strain_pct
strain_plus_mid_inferior_strain_pct
strain_plus_mid_inferoseptal_strain_pct
strain_plus_apical_anteroseptal_strain_pct
strain_plus_apical_anterior_strain_pct
strain_plus_apical_anterolateral_strain_pct
strain_plus_apical_inferolateral_strain_pct
strain_plus_apical_inferior_strain_pct
strain_plus_apical_inferoseptal_strain_pct

strain rate cols: 18
strain_plus_basal_anteroseptal_strain_rate_1_s
strain_plus_basal_anterior_strain_rate_1_s
strain_plus_basal_anterolateral_strain_rate_1_s
strain_plus_basal_inferolateral_strain_rate_1_s
strain_plus_basal_inferior_strain_rate_1

,count,mean,std,min,25%,50%,75%,max
echo_lv_strain_plus_cv_pct,32.0,-25.853204,11.408287,-57.514742,-31.766690,-23.442942,-18.204479,-10.835993
new_strain_plus_cv_pct,32.0,25.853204,11.408287,10.835993,18.204479,23.442942,31.766690,57.514742
echo_lv_strain_rate_cv_pct,32.0,-44.124666,21.598615,-119.367589,-49.680147,-40.325320,-29.148835,-19.319426
new_strain_rate_cv_pct,32.0,44.124666,21.598615,19.319426,29.148835,40.325320,49.680147,119.367589


,echo_lv_strain_plus_cv_pct,new_strain_plus_cv_pct,echo_lv_strain_rate_cv_pct,new_strain_rate_cv_pct
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN


In [3]:
x = df[strain_pct_cols].apply(pd.to_numeric, errors="coerce")

display(
    x.notna().sum(axis=1).describe()
)

count    153.000000
mean       3.764706
std        7.344676
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max       18.000000
dtype: float64

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Ars\projects\university\lab_urfu_2026\false-lv-chords")
DATA_PATH = PROJECT_ROOT / "data" / "final_analysis_dataset_ver2.xlsx"

df = pd.read_excel(DATA_PATH)

# 18 segmental strain columns
strain_pct_cols = [
    c for c in df.columns
    if (
        "strain_plus_" in c.lower()
        and c.lower().endswith("_strain_pct")
    )
]

# 18 segmental strain-rate columns
strain_rate_cols = [
    c for c in df.columns
    if (
        "strain_plus_" in c.lower()
        and c.lower().endswith("_strain_rate_1_s")
    )
]

print("strain_pct_cols:", len(strain_pct_cols))
print("strain_rate_cols:", len(strain_rate_cols))

def calculate_segmental_cv_pct(data: pd.DataFrame, cols: list[str], min_segments: int = 16):
    x = data[cols].apply(pd.to_numeric, errors="coerce")
    
    n_segments = x.notna().sum(axis=1)
    mean = x.mean(axis=1, skipna=True)
    sd = x.std(axis=1, skipna=True)
    
    cv = sd / mean.abs() * 100
    cv = cv.replace([np.inf, -np.inf], np.nan)
    
    cv[n_segments < min_segments] = np.nan
    
    return cv, n_segments, mean, sd

new_strain_cv, n_strain_segments, mean_strain, sd_strain = calculate_segmental_cv_pct(
    df,
    strain_pct_cols,
    min_segments=16,
)

new_strain_rate_cv, n_strain_rate_segments, mean_strain_rate, sd_strain_rate = calculate_segmental_cv_pct(
    df,
    strain_rate_cols,
    min_segments=16,
)

check_df = pd.DataFrame({
    "old_echo_lv_strain_plus_cv_pct": df["echo_lv_strain_plus_cv_pct"],
    "new_echo_lv_strain_plus_cv_pct": new_strain_cv,
    "n_strain_segments": n_strain_segments,
    "mean_strain": mean_strain,
    "sd_strain": sd_strain,
    
    "old_echo_lv_strain_rate_cv_pct": df["echo_lv_strain_rate_cv_pct"],
    "new_echo_lv_strain_rate_cv_pct": new_strain_rate_cv,
    "n_strain_rate_segments": n_strain_rate_segments,
    "mean_strain_rate": mean_strain_rate,
    "sd_strain_rate": sd_strain_rate,
})

print("\nSegment availability:")
display(
    check_df[
        ["n_strain_segments", "n_strain_rate_segments"]
    ].describe().T
)

print("\nOld vs recalculated CV summary:")
display(
    check_df[
        [
            "old_echo_lv_strain_plus_cv_pct",
            "new_echo_lv_strain_plus_cv_pct",
            "old_echo_lv_strain_rate_cv_pct",
            "new_echo_lv_strain_rate_cv_pct",
        ]
    ].describe().T
)

print("\nRows with valid recalculated strain CV:")
display(
    check_df[
        check_df["new_echo_lv_strain_plus_cv_pct"].notna()
        | check_df["new_echo_lv_strain_rate_cv_pct"].notna()
    ].head(40)
)

strain_pct_cols: 18
strain_rate_cols: 18

Segment availability:


,count,mean,std,min,25%,50%,75%,max
n_strain_segments,153.0,3.764706,7.344676,0.0,0.0,0.0,0.0,18.0
n_strain_rate_segments,153.0,3.764706,7.344676,0.0,0.0,0.0,0.0,18.0



Old vs recalculated CV summary:


,count,mean,std,min,25%,50%,75%,max
old_echo_lv_strain_plus_cv_pct,32.0,-25.853204,11.408287,-57.514742,-31.766690,-23.442942,-18.204479,-10.835993
new_echo_lv_strain_plus_cv_pct,32.0,25.853204,11.408287,10.835993,18.204479,23.442942,31.766690,57.514742
old_echo_lv_strain_rate_cv_pct,32.0,-44.124666,21.598615,-119.367589,-49.680147,-40.325320,-29.148835,-19.319426
new_echo_lv_strain_rate_cv_pct,32.0,44.124666,21.598615,19.319426,29.148835,40.325320,49.680147,119.367589



Rows with valid recalculated strain CV:


,old_echo_lv_strain_plus_cv_pct,new_echo_lv_strain_plus_cv_pct,n_strain_segments,mean_strain,sd_strain,old_echo_lv_strain_rate_cv_pct,new_echo_lv_strain_rate_cv_pct,n_strain_rate_segments,mean_strain_rate,sd_strain_rate
107,-43.989970,43.989970,18,-14.494444,6.376102,-84.237433,84.237433,18,-0.994444,0.837694
108,-22.380823,22.380823,18,-28.466667,6.371074,-62.272730,62.272730,18,-1.061111,0.660783
110,-17.685571,17.685571,18,-28.994444,5.127833,-22.398869,22.398869,18,-1.322222,0.296163
111,-31.658932,31.658932,18,-38.555556,12.206277,-49.647938,49.647938,18,-1.583333,0.786092
112,-26.173717,26.173717,18,-24.172222,6.326769,-42.156095,42.156095,18,-1.305556,0.550371
113,-31.168895,31.168895,18,-19.511111,6.081398,-44.784364,44.784364,18,-0.916667,0.410523
115,-21.094764,21.094764,18,-23.555556,4.968989,-35.498152,35.498152,18,-1.294444,0.459504
116,-10.844729,10.844729,18,-28.422222,3.082313,-40.310660,40.310660,18,-1.772222,0.714394
117,-10.835993,10.835993,18,-20.972222,2.272549,-42.354248,42.354248,18,-1.250000,0.529428
118,-31.391124,31.391124,18,-20.961111,6.579928,-60.379046,60.379046,18,-1.172222,0.707777


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Ars\projects\university\lab_urfu_2026\false-lv-chords")
DATA_PATH = PROJECT_ROOT / "data" / "final_analysis_dataset_ver2.xlsx"

df = pd.read_excel(DATA_PATH)

strain_pct_cols = [
    c for c in df.columns
    if "strain_plus_" in c.lower() and c.lower().endswith("_strain_pct")
]

strain_rate_cols = [
    c for c in df.columns
    if "strain_plus_" in c.lower() and c.lower().endswith("_strain_rate_1_s")
]

def calculate_segmental_cv_pct(data, cols, min_segments=16):
    x = data[cols].apply(pd.to_numeric, errors="coerce")
    n_segments = x.notna().sum(axis=1)
    mean = x.mean(axis=1, skipna=True)
    sd = x.std(axis=1, skipna=True)

    cv = sd / mean.abs() * 100
    cv = cv.replace([np.inf, -np.inf], np.nan)
    cv[n_segments < min_segments] = np.nan

    return cv

df["echo_lv_strain_plus_cv_pct"] = calculate_segmental_cv_pct(
    df, strain_pct_cols, min_segments=16
)

df["echo_lv_strain_rate_cv_pct"] = calculate_segmental_cv_pct(
    df, strain_rate_cols, min_segments=16
)

display(
    df[
        [
            "echo_lv_strain_plus_cv_pct",
            "echo_lv_strain_rate_cv_pct",
        ]
    ].describe().T
)

df.to_excel(DATA_PATH, index=False)

print("Dataset overwritten:")
print(DATA_PATH)

,count,mean,std,min,25%,50%,75%,max
echo_lv_strain_plus_cv_pct,32.0,25.853204,11.408287,10.835993,18.204479,23.442942,31.766690,57.514742
echo_lv_strain_rate_cv_pct,32.0,44.124666,21.598615,19.319426,29.148835,40.325320,49.680147,119.367589


Dataset overwritten:
C:\Users\Ars\projects\university\lab_urfu_2026\false-lv-chords\data\final_analysis_dataset_ver2.xlsx


In [6]:
display(
    df[
        [
            "echo_lv_strain_plus_cv_pct",
            "echo_lv_strain_rate_cv_pct",
        ]
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
echo_lv_strain_plus_cv_pct,32.0,25.853204,11.408287,10.835993,18.204479,23.442942,31.766690,57.514742
echo_lv_strain_rate_cv_pct,32.0,44.124666,21.598615,19.319426,29.148835,40.325320,49.680147,119.367589
